# Train FER-2013 on Kaggle using the cloned project helpers

1. Upload this notebook to Kaggle. Enable **Internet** and a **GPU accelerator**.
2. Replace REPO_URL with your uploaded repository's HTTPS URL.
3. Add [msambare/fer2013](https://www.kaggle.com/datasets/msambare/fer2013)
   through **Add Input**. An optional kagglehub download is supported below.
4. Run all cells and download `fer2013_results.zip` from the Output panel.

The helper code is imported from the cloned `src/` directory. Training logic
is shared with `notebooks/train.ipynb`, rather than copied into this notebook.
The existing Market-1501 notebook was used only as a workflow reference.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlparse

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # EDIT THIS.
REPO_REF = ""  # Optional existing branch/tag. Empty = the repository default.
WORKSPACE_ROOT = Path("/kaggle/working")
REPO_ROOT = WORKSPACE_ROOT / "fer2013-project"

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Replace REPO_URL with the HTTPS URL of your uploaded project.")
if urlparse(REPO_URL).scheme != "https":
    raise ValueError("Use an HTTPS Git URL.")
if not WORKSPACE_ROOT.is_dir():
    raise RuntimeError("This notebook is intended for Kaggle: /kaggle/working is missing.")
if (REPO_ROOT / ".git").is_dir():
    origin = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], text=True,
    ).strip()
    if origin != REPO_URL:
        raise ValueError(f"Existing clone uses {origin}; use a fresh destination for {REPO_URL}.")
    print("Reusing the existing clone; no automatic pull or reset.")
else:
    command = ["git", "clone", "--depth", "1"]
    if REPO_REF:
        command += ["--branch", REPO_REF]
    subprocess.run(command + [REPO_URL, str(REPO_ROOT)], check=True)
if not (REPO_ROOT / "src" / "train.py").is_file():
    raise FileNotFoundError("The cloned repo must contain src/train.py and requirements.txt.")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True,
).strip()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)
print("Commit:", REPO_COMMIT)

In [ ]:
# Existing compatible Kaggle torch/torchvision installs already satisfy the
# minimum requirements, so pip does not upgrade them unnecessarily.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")],
    check=True,
)

In [ ]:
import json
import shutil
import torch
from IPython.display import Image, display

from src.data import CLASS_NAMES, build_data
from src.model import build_model, count_parameters
from src.train import TrainConfig, fit, seed_everything
from src.evaluate import plot_class_distribution, evaluate_checkpoint, compare_models
from src.report import build_report

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before full training.")

from src.data import find_data_root

DATA_ROOT_OVERRIDE = ""  # Optional exact path if more than one FER input exists.
DOWNLOAD_IF_MISSING = True
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = find_data_root(DATA_ROOT_OVERRIDE)
else:
    try:
        DATA_ROOT = find_data_root("/kaggle/input")
    except FileNotFoundError:
        if not DOWNLOAD_IF_MISSING:
            raise FileNotFoundError("Add msambare/fer2013 through Kaggle Add Input, then rerun.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        # Use the HTTP downloader, including during non-interactive Kaggle runs.
        os.environ["DISABLE_KAGGLE_CACHE"] = "true"
        import kagglehub
        DATA_ROOT = find_data_root(kagglehub.dataset_download("msambare/fer2013"))

OUTPUT_ROOT = WORKSPACE_ROOT / "fer2013_results" / "outputs"
WEIGHTS_DIR = WORKSPACE_ROOT / "fer2013_results" / "weights"
REPORT_PATH = WORKSPACE_ROOT / "fer2013_results" / "report.pdf"
BATCH_SIZE = 128  # Reduce to 64 or 32 if GPU memory is insufficient.
NUM_WORKERS = 2
VAL_FRACTION = 0.10
SEED = 42
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "repo_commit.txt").write_text(REPO_COMMIT + "\n", encoding="utf-8")
print("Dataset:", DATA_ROOT)
print("Results:", OUTPUT_ROOT)

## 1. Inspect the data and the split

Both models use the same stratified train/validation split. Validation is held
out of the original `train/` directory, never from `test/`. The class weights
are computed **only from training labels**: square-root inverse frequency,
normalized to mean one. This gives rare classes more influence without the
very large weights of full inverse frequency.

Training adds mild flips/affine augmentation. Every split uses grayscale
48x48 and normalization `(pixel / 255 - 0.5) / 0.5`. The app reuses this code.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED)
data = build_data(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                  val_fraction=VAL_FRACTION, seed=SEED)
counts = plot_class_distribution(data, OUTPUT_ROOT / "class_distribution.png")
data.save_manifest(OUTPUT_ROOT / "split_manifest.json")
print("Resolved data directory:", data.root)
print(json.dumps(counts, indent=2))
print("Class order:", CLASS_NAMES)
print("Training class weights:", data.class_weights.tolist())
display(Image(filename=str(OUTPUT_ROOT / "class_distribution.png")))

## 2. Inspect M1 and M2

**M1 matches the supplied architecture.** Each block contains one padded 3x3
convolution, BatchNorm, ReLU, 2x2 max pool, and ordinary dropout. Channels:
32, 64, 128, 256; dropout: 0.10, 0.15, 0.20, 0.25. Its spatial sizes are
24, 12, 6, 3, so flattening produces 2304 features. The head is
Linear(2304,256), BatchNorm, ReLU, Dropout(0.50), Linear(256,7).

**M2 is ImageNet-pretrained ResNet-18.** The first convolution accepts one
grayscale channel; its RGB filters are summed, its stride becomes 1, and the
initial max pool is removed to retain detail at 48x48. The final layer becomes
Dropout(0.30) + Linear(512,7). All layers are fine-tuned, with a smaller learning
rate in the backbone. The shared FER normalization differs from stock ImageNet
preprocessing; fine-tuning adapts the network to that change.

Both networks return **logits**, not softmax, because CrossEntropyLoss applies
log-softmax itself. Inference applies softmax once to display confidence.

In [ ]:
torch.set_num_threads(4)
for name in ["cnn", "transfer"]:
    # Architecture inspection does not download pretrained weights.
    model = build_model(name, pretrained=False).eval()
    with torch.inference_mode():
        scores = model(torch.zeros(2, 1, 48, 48))
    print(f"{name}: {count_parameters(model):,} parameters; output {tuple(scores.shape)}")
    del model

## 3. Training configuration

AdamW, cosine learning-rate decay, weighted cross entropy with mild label
smoothing, gradient clipping and optional CUDA mixed precision. Best weights
are selected by **validation accuracy**, breaking ties with validation macro F1.
Early stopping uses the same rule. Test metrics do not influence training.

Loss plots show plain cross entropy for both splits; `train_objective_loss`
separately records the weighted/smoothed optimization objective in history.csv.
Training loss includes augmentation and dropout, so it can exceed validation loss.

Each `fit` call starts a fresh run and overwrites its model output directory.
Use a new OUTPUT_ROOT for a separate experiment. Best/last files are portable
inference checkpoints, not optimizer-resume checkpoints.

In [ ]:
cnn_config = TrainConfig(
    model_name="cnn", epochs=80, learning_rate=1e-3,
    weight_decay=1e-4, patience=18, label_smoothing=0.05,
    use_class_weights=True, pretrained=False, seed=SEED, device="auto",
)
transfer_config = TrainConfig(
    model_name="transfer", epochs=20, learning_rate=1e-3, backbone_lr=1e-4,
    weight_decay=1e-4, patience=12, label_smoothing=0.05,
    use_class_weights=True, pretrained=True, seed=SEED, device="auto",
)
# ResNet pretrained weights download on first use: Internet must be enabled.
print(cnn_config)
print(transfer_config)

## 4. Train M1 from scratch

Fix the training recipe before opening final test results. If a CUDA
out-of-memory error occurs, reduce BATCH_SIZE above and rebuild `data`.

In [ ]:
cnn_run = fit(cnn_config, data, OUTPUT_ROOT / "cnn")
print("Best M1 checkpoint:", cnn_run["checkpoint"])

## 5. Fine-tune M2 (ResNet-18)

In [ ]:
# Rebuild loaders so M2 starts from the same seeded shuffle/split as M1.
data = build_data(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                  val_fraction=VAL_FRACTION, seed=SEED)
transfer_run = fit(transfer_config, data, OUTPUT_ROOT / "transfer")
print("Best M2 checkpoint:", transfer_run["checkpoint"])

## 6. Final test evaluation and M3 comparison

Evaluate the best validation checkpoint for each model. Export accuracy,
macro F1, parameter counts, and CPU forward FPS, along with the full confusion
matrix and per-class report. CPU timing is **model-only**, batch one, float32,
four threads; it excludes face detection, preprocessing and camera I/O.
The application's displayed FPS measures capture + processing separately.

The lab targets (60% M1, 65% M2) are checked against measured test accuracy;
the code does not assume they have been achieved.

In [ ]:
all_metrics = []
for name in ["cnn", "transfer"]:
    metrics = evaluate_checkpoint(
        OUTPUT_ROOT / name / "best.pt", data, device="auto",
    )
    all_metrics.append(metrics)
    shutil.copy2(OUTPUT_ROOT / name / "best.pt", WEIGHTS_DIR / f"{name}_best.pt")
    print(json.dumps(metrics, indent=2))

comparison = compare_models(all_metrics, OUTPUT_ROOT)
print(f"{'Model':<10} {'Accuracy':>10} {'Macro F1':>10} {'Parameters':>12} {'CPU FPS':>10}")
for row in comparison:
    print(f"{row['model']:<10} {row['test_accuracy']:>10.2%} "
          f"{row['test_macro_f1']:>10.4f} {row['parameters']:>12,} {row['cpu_model_fps']:>10.1f}")

## 7. M4 and M5: inspect plots and mistakes

Each model exports up to ten errors, covering different true classes when
possible. The probability-based comments in `error_analysis.md` are a starting
point: inspect the faces and add image-specific observations about pose,
occlusion, lighting or ambiguous expressions. They are not invented visual
explanations. If fewer than ten mistakes exist, all available mistakes are shown.

In [ ]:
for name in ["cnn", "transfer"]:
    print("\n", name.upper())
    for filename in ["training_curves.png", "confusion_matrix.png", "misclassified.png"]:
        path = OUTPUT_ROOT / name / filename
        if path.exists():
            display(Image(filename=str(path)))
    print((OUTPUT_ROOT / name / "error_analysis.md").read_text(encoding="utf-8"))

## 8. Generate the six-page report

This uses actual saved results. Until you provide a phone-camera screenshot
and finish your visual error analysis, treat the PDF as a draft. Run the app
locally and press **s** to save `outputs/demo_screenshot.jpg`, then regenerate
the report. Edit `src/report.py` to incorporate your own written analysis.

In [ ]:
screenshot = OUTPUT_ROOT / "demo_screenshot.jpg"
report_path = build_report(
    OUTPUT_ROOT, REPORT_PATH,
    screenshot=screenshot if screenshot.exists() else None,
    author="YOUR NAME / STUDENT ID",
)
print("Report:", report_path)
print("Weights:", WEIGHTS_DIR)

## 9. Package results for download

The archive contains both best checkpoints, all evaluation artifacts and the
generated report draft. After saving a Kaggle notebook version with outputs,
download the executed notebook too. The supplied local notebook remains
unexecuted until you run it; keep an executed copy for the PDF's submission
requirement. Copy the downloaded weights into the local project's `weights/`.

In [ ]:
from shutil import make_archive

archive = make_archive(
    str(WORKSPACE_ROOT / "fer2013_results"), "zip",
    root_dir=WORKSPACE_ROOT / "fer2013_results",
)
print("Download from the Kaggle Output panel:", archive)